# Rio Mora near Terrero, NM: sequential Bayesian calibration and forecasting

Distributed rainfall–runoff model for a 53.2 mi² forested basin in the Sangre de Cristo Mountains, New Mexico, where floods are driven by monsoon convection. Hourly MRMS rainfall on each radar cell (about 1 km) is routed to the gauge through a hillslope linear reservoir and an advection–diffusion channel kernel. The model parameters have normal priors and are updated with Bayes' rule one storm at a time, in date order.

Each storm is first forecast from the current prior, with a 90 % predictive interval, before its hydrograph is used. The storm is then used to update the parameters, and the posterior becomes the prior for the next storm.

The first `N_CALIBRATE` storms (default 30) are used for calibration only. Plots and scores start with the first storm after the calibration storms.

Required modules for this notebook: `distributed_iuh.py`, `nash_iuh.py`, `sequential_bayes.py`

1. Settings
2. Watershed, streamflow and event selection
3. Radar rainfall and event screening
4. Sequential Bayesian update
5. Parameter posteriors and forecast skill
6. Cell routing
7. Forecasts and posterior fits after calibration
8. Updating with a new storm

## 1. Settings

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nash_iuh as m
import distributed_iuh as d
import sequential_bayes as sb

SITE = "08377900"              # Rio Mora near Terrero, NM (Sangre de Cristo Mountains)
DRAINAGE_AREA_MI2 = 53.2       # USGS published drainage area; None = look it up
YEARS = range(2015, 2022)      # ends before the 2022 Hermits Peak-Calf Canyon fire
MONTHS = (7, 8, 9, 10)         # monsoon season, after April-June snowmelt
DT_MIN = 15
DT = DT_MIN / 60.0             # model time step (h)

MAX_EVENTS = 80                # number of largest storms retained (limits radar download); None = all
CACHED_ONLY = False            # True = no downloads: cached flow and radar only; storms without cached radar are excluded
N_CALIBRATE = 30               # calibration storms (date order): used in the update, not plotted or scored
MIN_RISE_CFS = None            # minimum peak rise (cfs); None = set from the flow record
RADAR_SOURCE = "auto"          # 'auto' | 'iem' (Oct 2014 on) | 'aws' (Oct 2020 on)
A_IMP = None                   # impervious fraction: None = estimated; or a fixed value, e.g. 0.03
HILL_EXP = 1.0                 # hillslope residence-time exponent: 1 = k_i proportional to distance
                               # to the nearest stream; 0 = uniform k_hill
KB_DAYS = None                 # baseflow recession constant (days); None = estimated from the record

# Bayesian update
PRIOR = {}                     # prior overrides, e.g. {"c_ms": (0.8, 0.4)}; others use the defaults
TAU_IA = 0.35                  # sd (in) of storm Ia about the Ia-wetness regression
DRIFT = 0.0                    # covariance inflation per storm (fraction of the prior); 0 = static parameters
N_SAMPLES = 30                 # parameter samples per 90 % interval; 0 = no intervals
REUSE_SAVED_RUN = True         # load a saved run with the same storms and settings

DATA = f"data/USGS-{SITE}"
FIG = f"{DATA}/report_figures"   # figures named as in the report (copy to the report's figures/ folder)
TAG = "rio_mora"
os.makedirs(FIG, exist_ok=True)
pd.set_option("display.precision", 3)

## 2. Watershed, streamflow and event selection

Gauge metadata, basin boundary and NHDPlus flowlines are retrieved from the USGS Water Data API and the Network Linked Data Index (NLDI). The flowlines form a graph, and Dijkstra's algorithm gives the channel distance from every stream vertex to the gauge.

Discharge is the 15-minute record, with gaps of up to 2 h interpolated. The baseflow recession constant $k_b$ is the median daily recession constant over recession periods (at least 5 days of falling flow, first 3 days excluded). It is fixed, not estimated in the update.

Storm events are peaks in `MONTHS` with a rise above the threshold, separated by at least 24 h. Each window starts at the preceding trough (not before the previous peak) and ends 5 days after the peak or at the next event. The `MAX_EVENTS` largest events are retained.

In [ ]:
info = d.fetch_site_info(SITE, DATA)
paths = {k: f"{DATA}/nldi_{k}.geojson" for k in ("site", "basin", "flowlines")}
if not all(os.path.exists(p_) for p_ in paths.values()):
    d.fetch_nldi_geometry(SITE, DATA)
outlet, ring, lines = d.load_point(paths["site"]), d.load_ring(paths["basin"]), d.load_lines(paths["flowlines"])
net = d.build_channel_network(lines, outlet)
AREA_MI2 = float(DRAINAGE_AREA_MI2 or info.get("drainage_area_mi2") or d.polygon_area_mi2(ring))
bbox = (ring[:, 0].min() - 0.03, ring[:, 1].min() - 0.03, ring[:, 0].max() + 0.03, ring[:, 1].max() + 0.03)
print(f"USGS {SITE}: {info.get('name')}, {AREA_MI2:.1f} mi2, longest channel path "
      f"{np.nanmax(net.dist_km[net.reachable]):.1f} km")

flow_raw = d.load_or_fetch_flow(SITE, YEARS, f"{DATA}/flow_iv.csv", download=not CACHED_ONLY)
grid = pd.date_range(flow_raw.index.min().ceil("15min"), flow_raw.index.max().floor("15min"), freq="15min")
flow = flow_raw.reindex(flow_raw.index.union(grid)).interpolate(method="time", limit=8).reindex(grid)
KB = KB_DAYS or d.estimate_kb_days(flow_raw)
print(f"flow {flow.index[0]:%Y-%m-%d} to {flow.index[-1]:%Y-%m-%d}; k_b = {KB:.1f} days")

flow_season = flow[flow.index.month.isin(MONTHS)]     # monsoon months only, so snowmelt does not raise the threshold
rise = MIN_RISE_CFS or max(3 * flow_season.median(), 0.05 * flow_season.quantile(0.99))
events = m.find_events(flow.dropna(), None, AREA_MI2, min_rise_cfs=rise, months=MONTHS)
nxt = events.start.shift(-1).fillna(events.end.iloc[-1] + pd.Timedelta(days=5))
events["end"] = np.minimum(events.peak_time + pd.Timedelta(days=5), nxt - pd.Timedelta(hours=1))
events = events[events.end > events.peak_time + pd.Timedelta(hours=6)]
events = events.sort_values("rise_cfs", ascending=False).head(MAX_EVENTS or len(events)).sort_values("start").reset_index(drop=True)
print(f"{len(events)} storm windows (minimum rise {rise:,.0f} cfs)")

Elevation map (USGS 3DEP). Elevation is used for this figure only.

In [ ]:
DEM_FILE = f"{DATA}/dem.npz"
if not os.path.exists(DEM_FILE):
    d.fetch_dem(bbox, DEM_FILE)
ax = d.plot_topography(d.load_dem(DEM_FILE), ring, lines, outlet, title=f"{info.get('name', 'USGS ' + SITE)}")
ax.figure.savefig(f"{FIG}/elevation_{TAG}.png", dpi=250, bbox_inches="tight")
plt.show()

## 3. Radar rainfall and event screening

Hourly MRMS gauge-corrected rainfall is downloaded for the storm windows only and cached; an interrupted download resumes. With `CACHED_ONLY = True` nothing is downloaded, and storms without cached radar are excluded. For each radar cell the code computes the area inside the basin $A_i$, the channel distance to the gauge $L_i$, and the distance to the nearest stream $L_{hill,i}$.

A storm is excluded if radar data are missing for more than 2 % of its hours, its runoff depth is below 0.005 in or its runoff ratio is outside 0.005–0.95, the lag from maximum rainfall to peak is outside 0–25 h, or the window starts on the falling limb of an earlier event. Hourly cell rainfall is resampled to the 15-minute model step.

In [ ]:
RADAR_FILE = f"{DATA}/mrms_basin.npz"
if CACHED_ONLY:
    if not os.path.exists(RADAR_FILE):
        raise FileNotFoundError(f"{RADAR_FILE} not found; run once with CACHED_ONLY = False")
    cube = d.RadarCube.load(RADAR_FILE)
    print(f"CACHED_ONLY: {len(cube.times):,} cached radar hours, no download")
else:
    cube = d.download_radar(d.event_hours(events, lead_h=12), bbox, RADAR_FILE, source=RADAR_SOURCE)
cells = d.basin_cells(cube.lon1d, cube.lat1d, ring, net)

radar_avg = d.hourly_to_steps(cube.basin_average(cells), flow.index).fillna(0.0)
pieces, dropped = [], []
for _, e in events.iterrows():
    if d.radar_coverage(cube, e.start - pd.Timedelta(hours=6), e.end) < 0.98:
        dropped.append((e.start, "not in radar cache" if CACHED_ONLY else "radar coverage < 98 %")); continue
    r_, q_ = m.extract_event(radar_avg, flow, e.start, e.end, rain_lead_h=6)
    if r_.sum() <= 0 or q_.isna().any():
        dropped.append((e.start, "no rainfall or flow gap")); continue
    f_ = m.event_pieces(r_, q_, AREA_MI2, DT)
    if not (f_.runoff_in > 0.005 and 0.005 < f_.runoff_ratio < 0.95):
        dropped.append((e.start, f"runoff ratio {f_.runoff_ratio:.3f}")); continue
    pieces.append(f_)
lag = m.event_response_lags(pieces).lag_maxrain_to_peak_h.values
dropped += [(f_.time[0], f"rainfall-to-peak lag {l_:.1f} h") for f_, l_ in zip(pieces, lag) if not 0 <= l_ <= 25]
pieces = [f_ for f_, l_ in zip(pieces, lag) if 0 <= l_ <= 25]
storms = d.make_dist_events(pieces, cube.cell_rain(cells), DT_MIN)
share_below = d.baseflow_check(storms, KB, DT).share_below_recession.values
dropped += [(e.time[0], f"falling-limb start ({s_:.0%} of window below recession)")
            for e, s_ in zip(storms, share_below) if s_ > 0.10]
storms = [e for e, s_ in zip(storms, share_below) if s_ <= 0.10]

print(f"{len(cells)} radar cells, {cells.area_mi2.sum():.1f} mi2")
print(f"{len(storms)} storms retained, {len(dropped)} excluded")
if len(storms) <= N_CALIBRATE:
    print(f"warning: {len(storms)} storms <= N_CALIBRATE = {N_CALIBRATE}; the first half will be used for calibration "
          f"(increase MAX_EVENTS or YEARS, or reduce N_CALIBRATE)")
ax = d.plot_watershed_map(ring, lines, outlet, cells=cells, color="L_channel_km",
                          title="Radar cells: channel distance to the gauge")
plt.show()
pd.DataFrame(dropped, columns=["storm start", "reason excluded"])

## 4. Sequential Bayesian update

The parameter vector holds seven model parameters ($c$, $D$, $k_{hill}$, $f_c$, $a$, $\beta$, $k_{gw}$) and the two coefficients of a regression for the initial abstraction on antecedent wetness:
$$I_{a,k} \sim N\big(I_{a,mean} + I_{a,slope}\, z_k,\ \tau^2\big), \qquad z_k = \text{standardised } \log_{10} Q_{0,k}.$$
All nine have normal priors, printed below ($D$, $k_{hill}$ and $k_{gw}$ in $\log_{10}$).

For each storm, in date order:
1. Forecast with the prior mean, $I_a$ from the regression, and a 90 % predictive interval from parameter and $I_a$ samples combined with the typical model error.
2. Update with Bayes' rule, $p(\theta \mid y_{1:k}) \propto p(y_k \mid \theta)\, p(\theta \mid y_{1:k-1})$, using a Laplace (normal) approximation at the maximum a posteriori (MAP) estimate. The posterior hydrograph has its own 90 % interval.
3. Use the posterior as the prior for the next storm.

Calibration storms (the first `N_CALIBRATE`) are processed in the same way but are not plotted or scored.

The run is saved to disk; with `REUSE_SAVED_RUN = True` a matching run is loaded instead of recomputed. A full run with 30 samples per interval takes a few minutes.

In [ ]:
names_ = sb.param_names(A_IMP, KB is not None)[0]
prior_ = {**sb.default_prior(names_), **PRIOR}
print("Normal priors (mean, sd), in the estimation units:")
for k_ in names_:
    print(f"  {k_:13s} N({prior_[k_][0]:7.3f}, {prior_[k_][1]:.3f})")
print()
seq = sb.run_saved(storms, cells, DT, folder=f"{DATA}/runs", label="sequential", reuse=REUSE_SAVED_RUN,
                   gauge_area_mi2=AREA_MI2, kb_days=KB, a_imp=A_IMP, hill_exp=HILL_EXP, prior=prior_, tau_ia=TAU_IA,
                   drift=DRIFT, n_samples=N_SAMPLES, verbose=True)
N_CAL = min(N_CALIBRATE, len(seq.events) // 2)         # too few storms: calibrate on the first half
if N_CAL < N_CALIBRATE:
    print(f"\n{len(seq.events)} storms: calibration set reduced to the first {N_CAL}")
TEST = list(range(N_CAL, len(seq.events)))             # storms plotted and scored (0-based, date order)
if N_CAL:
    print(f"\ncalibration: storms 1-{N_CAL} ({seq.events[0].time[0]:%Y-%m-%d} to {seq.events[N_CAL - 1].time[0]:%Y-%m-%d}); "
          f"evaluation: storms {N_CAL + 1}-{len(seq.events)} ({len(TEST)} storms)")
print("\nPrior and final posterior, physical units (90 % intervals):")
sb.posterior_table(seq).round(3)

## 5. Parameter posteriors and forecast skill

The first figure shows the posterior mean and 90 % interval of each parameter after each update (storm 0 is the prior), with the calibration storms shaded. A narrowing interval indicates that the parameter is identified by the data.

The second figure gives forecast skill for the evaluation storms: NSE of the forecast, timing efficiency (TE) of the forecast, and NSE of the posterior fit. The right panel shows posterior standard deviation as a percentage of the prior standard deviation.

The third figure shows the posterior $I_a$ of each storm against pre-storm flow, with the regression used in the forecasts.

In [ ]:
sb.plot_parameter_paths(seq, n_calibrate=N_CAL)
plt.gcf().savefig(f"{FIG}/parameter_paths_{TAG}.pdf", bbox_inches="tight"); plt.show()
sb.plot_forecast_skill(seq, n_calibrate=N_CAL)
plt.gcf().savefig(f"{FIG}/forecast_skill_{TAG}.pdf", bbox_inches="tight"); plt.show()
sb.plot_ia_relation(seq)
plt.gcf().savefig(f"{FIG}/ia_regression_{TAG}.pdf", bbox_inches="tight"); plt.show()

h_ = seq.history.iloc[TEST]
print(f"{len(h_)} evaluation storms: median forecast NSE {h_.NSE_forecast.median():.2f}, "
      f"median forecast TE {h_.TE_forecast.median():.2f}, median posterior NSE {h_.NSE_after_update.median():.2f}")
if "obs_inside_90%_band" in h_:
    print(f"coverage of the 90 % forecast interval: {100 * h_['obs_inside_90%_band'].mean():.0f} % (nominal 90 %)")

In [ ]:
print(f"median posterior TE {np.median([d.timing_metrics(seq.events[k].Q_obs, seq.forecasts[k]['Q_after'], DT, base=seq.forecasts[k]['recession'])['TE'] for k in TEST]):.2f}")

## 6. Cell routing

All figures use the final posterior mean parameters.

Hillslope residence time. Top: distance from each radar cell to the nearest mapped stream. Bottom: hillslope residence time
$$k_i = k_{hill}\left(\frac{\max(L_{hill,i},\ 0.25\text{ km})}{\bar L_{hill}}\right)^{\texttt{HILL\_EXP}}.$$
With `HILL_EXP = 1` (default) the residence time is proportional to the distance to the stream. With `HILL_EXP = 0` all cells use $k_{hill}$, and the map shows the values for `HILL_EXP = 1` for reference.

Routing of a single cell: (a) the cell, its hillslope path and its channel path to the gauge; (b) its Green's function; (c) the width function; (d) the geomorphologic IUH (GIUH) with the contribution of near, middle and far thirds of the basin. `CELL = None` selects a cell far from the streams, about three quarters of the way up the network; otherwise give a row index of `cells`.

In [ ]:
p = seq.params()
if HILL_EXP > 0:
    k_i = seq.router.k_cells(p)
    k_title = f"Hillslope residence time $k_i$ (HILL_EXP = {HILL_EXP:g})"
else:                           # uniform k_hill in the model; map shows HILL_EXP = 1 for reference
    k_i = d.Router(cells, DT, hill_exp=1.0).k_cells(p)
    k_title = "Hillslope residence time $k_i$ for HILL_EXP = 1 (reference; model uses uniform $k_{hill}$)"
print(f"k_hill = {p.k_hill_h:.2f} h; mapped k_i from {k_i.min():.2f} to {k_i.max():.2f} h "
      f"(area-weighted mean {np.average(k_i, weights=cells.area_mi2):.2f} h)")
fig, axs = plt.subplots(2, 1, figsize=(12, 10))
d.plot_watershed_map(ring, lines, outlet, cells=cells, color="L_hill_km", ax=axs[0],
                     title="Distance to the nearest stream, $L_{hill}$")
d.plot_watershed_map(ring, lines, outlet, cells=cells.assign(k_i_h=k_i), color="k_i_h", ax=axs[1], title=k_title)
fig.tight_layout()
fig.savefig(f"{DATA}/hillslope_time_map.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
CELL = None                     # None = automatic selection; or a row index of `cells`
fig = d.plot_routing_schematic(cells, net, p, ring, lines, outlet, cell=CELL, dt=DT, router=seq.router)
fig.savefig(f"{FIG}/cell_routing_{TAG}.pdf", bbox_inches="tight")
plt.show()

## 7. Forecasts and posterior fits after calibration

Evaluation storms in date order, one panel each. Flows are event flow (observed flow minus the pre-storm recession). Black: observed. Orange: forecast from the prior, with its 90 % predictive interval shaded. Green dashed: posterior fit after the update. The inverted bars give basin-mean rainfall (light) and the effective rainfall of the forecast (dark). Panel titles give storm date, rainfall depth P, runoff depth Q and forecast NSE.

The grid is saved as `storm_grid_<basin>.pdf` in the `report_figures` folder. Set `GRID_ROWS` to split it over several figures.

In [ ]:
GRID_COLS = 4                    # panels per row
GRID_ROWS = None                 # None = single figure; or rows per figure, e.g. 3
per_fig = len(TEST) if GRID_ROWS is None else GRID_COLS * GRID_ROWS
for f_ in range(0, len(TEST), per_fig):
    ks_ = TEST[f_:f_ + per_fig]
    fig = sb.plot_storm_grid(seq, ks_, ncols=GRID_COLS,
                             title=f"Storms {ks_[0] + 1}–{ks_[-1] + 1} of {len(seq.events)}: forecast (orange, 90 % interval), "
                                   f"observed (black), posterior after update (green dashed)")
    name = "storm_grid" if per_fig >= len(TEST) else f"storm_grid_{f_ // per_fig + 1}"
    fig.savefig(f"{FIG}/{name}_{TAG}.pdf", bbox_inches="tight")
    plt.show()

Individual storms with rainfall maps. Orange: forecast from the prior (parameters from earlier storms, $I_a$ from the regression), with its 90 % predictive interval. Green: posterior fit with the storm's posterior $I_a$; the hatched area is its 90 % interval. The map gives storm-total rainfall on each radar cell.

`SHOW` restricts the output to a list of storm numbers (calibration storms included). Each figure is saved as `storm_<number>_<basin>.pdf` in the `report_figures` folder.

In [ ]:
SHOW = None                     # None = all evaluation storms; or storm numbers, e.g. [31, 40, 52]
for k in (TEST if SHOW is None else [s_ - 1 for s_ in SHOW]):
    ev = seq.events[k]
    sb.plot_forecast_with_map(seq, seq.forecasts[k], ev, lines, outlet,
                              title=f"Storm {k + 1} of {len(seq.events)}, {ev.time[0]:%Y-%m-%d}: "
                                    f"forecast from {k} prior storm{'s' * (k != 1)}")
    plt.gcf().savefig(f"{FIG}/storm_{k + 1:02d}_{TAG}.pdf", bbox_inches="tight")
    plt.show()

## 8. Updating with a new storm

A new storm is prepared as in section 3 (add its year to `YEARS`, rerun sections 2–3 and select it from `storms`), then forecast and used for the update:

```python
fc = seq.update_storm(new_storm)       # forecast, then update
sb.plot_forecast_with_map(seq, fc, new_storm, lines, outlet)
```

Storms must be added in date order. For a forecast without an update, use `fc = seq.predict(new_storm, n_samples=N_SAMPLES)`.